<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Custom embedddings con Gensim



### Objetivo
El objetivo es utilizar documentos / corpus para crear embeddings de palabras basado en ese contexto. Se utilizará canciones de bandas para generar los embeddings, es decir, que los vectores tendrán la forma en función de como esa banda haya utilizado las palabras en sus canciones.

In [34]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import multiprocessing
try:
  from gensim.models import Word2Vec
except:
  !pip install gensim
  from gensim.models import Word2Vec

### Datos
Utilizaremos como dataset canciones de bandas de habla inglesa.

In [35]:
# Descargar la carpeta de dataset
import os
import platform
if os.access('./songs_dataset', os.F_OK) is False:
    if os.access('songs_dataset.zip', os.F_OK) is False:
        if platform.system() == 'Windows':
            !curl https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip -o songs_dataset.zip
        else:
            !wget songs_dataset.zip https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
    !unzip -q songs_dataset.zip
else:
    print("El dataset ya se encuentra descargado")

El dataset ya se encuentra descargado


In [36]:
# Posibles bandas
os.listdir("./songs_dataset/")

['prince.txt',
 'dickinson.txt',
 'notorious-big.txt',
 'beatles.txt',
 'bob-dylan.txt',
 'bjork.txt',
 'johnny-cash.txt',
 'disney.txt',
 'janisjoplin.txt',
 'kanye.txt',
 'bob-marley.txt',
 'leonard-cohen.txt',
 '.DS_Store',
 'ludacris.txt',
 'adele.txt',
 'alicia-keys.txt',
 'joni-mitchell.txt',
 'amy-winehouse.txt',
 'lorde.txt',
 'rihanna.txt',
 'Kanye_West.txt',
 'nirvana.txt',
 'cake.txt',
 'bieber.txt',
 'notorious_big.txt',
 'missy-elliott.txt',
 'dolly-parton.txt',
 'jimi-hendrix.txt',
 'songs_dataset.zip',
 'michael-jackson.txt',
 'al-green.txt',
 'lil-wayne.txt',
 'lady-gaga.txt',
 'lin-manuel-miranda.txt',
 'nursery_rhymes.txt',
 'dj-khaled.txt',
 'radiohead.txt',
 'patti-smith.txt',
 'blink-182.txt',
 'Lil_Wayne.txt',
 'dr-seuss.txt',
 'r-kelly.txt',
 'drake.txt',
 'britney-spears.txt',
 'bruce-springsteen.txt',
 'nicki-minaj.txt',
 'kanye-west.txt',
 'paul-simon.txt',
 'nickelback.txt',
 'eminem.txt',
 'bruno-mars.txt']

In [37]:
# Armar el dataset utilizando salto de línea para separar las oraciones/docs
df = pd.read_csv('./songs_dataset/beatles.txt', sep='/n', header=None)
df.head()

/var/folders/ff/89s4xv1x5210pv23kt90w28c0000gn/T/ipykernel_27087/3981868439.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv('./songs_dataset/beatles.txt', sep='/n', header=None)


,0
0,"Yesterday, all my troubles seemed so far away"
1,Now it looks as though they're here to stay
2,"Oh, I believe in yesterday Suddenly, I'm not h..."
3,There's a shadow hanging over me.
4,"Oh, yesterday came suddenly Why she had to go ..."


In [38]:
print("Cantidad de documentos:", df.shape[0])

Cantidad de documentos: 1846


### 1 - Preprocesamiento

In [39]:
try:
  from tensorflow.keras.preprocessing.text import text_to_word_sequence
except:
  !pip install tensorflow-macos
  !pip install tensorflow-metal
  from tensorflow.keras.preprocessing.text import text_to_word_sequence

In [40]:
sentence_tokens = []
# Recorrer todas las filas y transformar las oraciones
# en una secuencia de palabras (esto podría realizarse con NLTK o spaCy también)
for _, row in df[:None].iterrows():
    sentence_tokens.append(text_to_word_sequence(row[0]))

In [41]:
# Demos un vistazo
sentence_tokens[:2]

[['yesterday', 'all', 'my', 'troubles', 'seemed', 'so', 'far', 'away'],
 ['now', 'it', 'looks', 'as', 'though', "they're", 'here', 'to', 'stay']]

### 2 - Crear los vectores (word2vec)

In [42]:
from gensim.models.callbacks import CallbackAny2Vec
# Durante el entrenamiento gensim por defecto no informa el "loss" en cada época
# Sobrecargamos el callback para poder tener esta información
class callback(CallbackAny2Vec):
    """
    Callback to print loss after each epoch
    """
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss after epoch {}: {}'.format(self.epoch, loss))
        else:
            print('Loss after epoch {}: {}'.format(self.epoch, loss- self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [43]:
# Crearmos el modelo generador de vectores
# En este caso utilizaremos la estructura modelo Skipgram
w2v_model = Word2Vec(min_count=5,    # frecuencia mínima de palabra para incluirla en el vocabulario
                     window=2,       # cant de palabras antes y desp de la predicha
                     vector_size=300,       # dimensionalidad de los vectores
                     negative=20,    # cantidad de negative samples... 0 es no se usa
                     workers=1,      # si tienen más cores pueden cambiar este valor
                     sg=1)           # modelo 0:CBOW  1:skipgram

In [44]:
# Obtener el vocabulario con los tokens
w2v_model.build_vocab(sentence_tokens)

In [45]:
# Cantidad de filas/docs encontradas en el corpus
print("Cantidad de docs en el corpus:", w2v_model.corpus_count)

Cantidad de docs en el corpus: 1846


In [46]:
# Cantidad de words encontradas en el corpus
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))

Cantidad de words distintas en el corpus: 445


### 3 - Entrenar embeddings

In [47]:
# Entrenamos el modelo generador de vectores
# Utilizamos nuestro callback
w2v_model.train(sentence_tokens,
                 total_examples=w2v_model.corpus_count,
                 epochs=20,
                 compute_loss = True,
                 callbacks=[callback()]
                 )

Loss after epoch 0: 113045.28125
Loss after epoch 1: 65966.59375
Loss after epoch 2: 65934.921875
Loss after epoch 3: 65718.390625
Loss after epoch 4: 63875.125
Loss after epoch 5: 64160.90625
Loss after epoch 6: 64080.375
Loss after epoch 7: 64814.96875
Loss after epoch 8: 62632.6875
Loss after epoch 9: 60452.6875
Loss after epoch 10: 59840.125
Loss after epoch 11: 58884.0625
Loss after epoch 12: 57716.0625
Loss after epoch 13: 56493.875
Loss after epoch 14: 55817.5
Loss after epoch 15: 55842.9375
Loss after epoch 16: 51722.625
Loss after epoch 17: 49858.125
Loss after epoch 18: 49592.0
Loss after epoch 19: 48960.375


(156986, 287740)

### 4 - Ensayar

In [48]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["darling"], topn=10)

[('pretty', 0.8954240083694458),
 ('sleep', 0.8665648698806763),
 ('help', 0.8439376950263977),
 ('cry', 0.8351278305053711),
 ('not', 0.8309565782546997),
 ('try', 0.8276928663253784),
 ('peace', 0.8144848346710205),
 ('little', 0.8140541911125183),
 ('twist', 0.8123903274536133),
 ('seems', 0.8079550266265869)]

In [49]:
# Palabras que MENOS se relacionan con...:
w2v_model.wv.most_similar(negative=["love"], topn=10)

[('shake', -0.22872473299503326),
 ('four', -0.2330225110054016),
 ('five', -0.2374648153781891),
 ('six', -0.23784106969833374),
 ('bang', -0.24832147359848022),
 ('our', -0.25538334250450134),
 ('day', -0.2689809799194336),
 ('going', -0.26920318603515625),
 ('here', -0.2699020802974701),
 ('three', -0.2838955223560333)]

In [50]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["four"], topn=10)

[('five', 0.9813734889030457),
 ('three', 0.9745770692825317),
 ('six', 0.9710825681686401),
 ('seven', 0.9584391117095947),
 ('two', 0.9517229199409485),
 ('sixty', 0.8990437984466553),
 ('one', 0.7951187491416931),
 ('crying', 0.7946332693099976),
 ('us', 0.7740050554275513),
 ("i'm", 0.7508416175842285)]

In [51]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["money"], topn=5)

[("can't", 0.9434012770652771),
 ('buy', 0.9397005438804626),
 ('much', 0.9033158421516418),
 ('just', 0.8509080410003662),
 ('hide', 0.8355337381362915)]

In [104]:
# Ensayar con una palabra que no está en el vocabulario:
try:
    resultado = w2v_model.wv.most_similar(negative=["diedaa"])
    print(resultado)
except KeyError:
    print("La palabra 'diedaa' no está en el vocabulario del modelo")

La palabra 'diedaa' no está en el vocabulario del modelo


In [53]:
# el método `get_vector` permite obtener los vectores:
vector_love = w2v_model.wv.get_vector("love")
print(vector_love)

[ 0.06138405  0.058809   -0.06370325  0.02445593 -0.20152055 -0.18612228
 -0.15284562  0.45487782 -0.04217936  0.03536211  0.13657948 -0.18519847
 -0.18126771  0.2214913  -0.3038063  -0.23970455  0.07094643 -0.05679133
 -0.05166036 -0.23843044 -0.08529771  0.195644   -0.07679128  0.03796851
  0.07516617 -0.04825819  0.07379003  0.10396867  0.00737886 -0.22764948
 -0.04567026  0.12936729  0.27786148  0.19387393 -0.13509513  0.20856662
  0.40916505 -0.00386539 -0.10631859 -0.09057057  0.02400651 -0.08005195
  0.1340099   0.08833113 -0.01894911  0.08592594 -0.15906131  0.10259284
  0.14459819 -0.12092229 -0.279187   -0.0406201   0.11382356  0.31366625
 -0.07409906  0.13977523  0.22791535  0.13209976 -0.01811203  0.0977301
  0.09249622 -0.14872141 -0.16348357 -0.13202755 -0.09834751  0.02714126
  0.16531599  0.26052776 -0.03259096 -0.02894498  0.11620875 -0.06974296
  0.09563834 -0.15276456  0.22071241  0.1599668   0.15890227 -0.04711496
 -0.12555523 -0.03993179 -0.10795247  0.01879167  0.

In [54]:
# el método `most_similar` también permite comparar a partir de vectores
w2v_model.wv.most_similar(vector_love)

[('love', 1.0000001192092896),
 ('babe', 0.9085149168968201),
 ('someone', 0.8886125683784485),
 ('need', 0.8827982544898987),
 ('nothing', 0.8740253448486328),
 ("didn't", 0.863835334777832),
 ("there's", 0.8526684641838074),
 ('you', 0.8456718921661377),
 ('feed', 0.8445045948028564),
 ('somebody', 0.8362783789634705)]

In [55]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["love"], topn=10)

[('babe', 0.9085149168968201),
 ('someone', 0.8886125683784485),
 ('need', 0.8827982544898987),
 ('nothing', 0.8740253448486328),
 ("didn't", 0.8638353943824768),
 ("there's", 0.8526685237884521),
 ('you', 0.8456718921661377),
 ('feed', 0.8445045948028564),
 ('somebody', 0.8362783789634705),
 ('buy', 0.8351733088493347)]

### 5 - Visualizar agrupación de vectores

In [56]:
from sklearn.decomposition import IncrementalPCA
from sklearn.manifold import TSNE
import numpy as np

def reduce_dimensions(model, num_dimensions = 2 ):

    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)

    tsne = TSNE(n_components=num_dimensions, random_state=0)
    vectors = tsne.fit_transform(vectors)

    return vectors, labels

In [57]:
# Graficar los embedddings en 2D
import plotly.graph_objects as go
import plotly.express as px

vecs, labels = reduce_dimensions(w2v_model)

MAX_WORDS=200
fig = px.scatter(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], text=labels[:MAX_WORDS])
fig.show(renderer="colab") # esto para plotly en colab

In [58]:
# Graficar los embedddings en 3D

vecs, labels = reduce_dimensions(w2v_model,3)

fig = px.scatter_3d(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], z=vecs[:MAX_WORDS,2],text=labels[:MAX_WORDS])
fig.update_traces(marker_size = 2)
fig.show(renderer="colab") # esto para plotly en colab

In [59]:
# También se pueden guardar los vectores y labels como tsv para graficar en
# http://projector.tensorflow.org/


vectors = np.asarray(w2v_model.wv.vectors)
labels = list(w2v_model.wv.index_to_key)

np.savetxt("vectors.tsv", vectors, delimiter="\t")

with open("labels.tsv", "w") as fp:
    for item in labels:
        fp.write("%s\n" % item)

In [60]:
print(f"Palabras únicas: {len(w2v_model.wv.index_to_key)}")

Palabras únicas: 445


### Consigna del desafío 2

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado**

Recuerden que su notebook de entrega debe poder correrse de inicio a fin sin la aparición de errores.

- Crear sus propios vectores con Gensim basado en lo visto en clase con otro artista del dataset Songs.
- Elegir términos de interés y buscar términos más similares y menos similares.
- Realizar una reduccion de dimensionalidad a los embeddings, llevándolos a 2 dimensiones. Graficar los embeddings proyectados y seleccionar una cantidad de términos (variable MAX_WORDS) de forma tal que la visualización sea adecuada.
- Inspeccionar el grafico y buscar pequeños grupos de palabras que puedan formarse. Interpretarlos e intentar obtener conclusiones. En lo posible, acompañar los grupos de palabras con capturas (y pegarlas en celdas de texto)

Vamos a ver que pasa con Radiohead

In [61]:
# Carga de datos de Radiohead
df_rh = pd.read_csv('songs_dataset/radiohead.txt', sep='/n', header=None, engine='python')


In [62]:
print("Cantidad de documentos:", df_rh.shape[0])

Cantidad de documentos: 2343


In [63]:
sentence_tokens = []
# Recorrer todas las filas y transformar las oraciones
# en una secuencia de palabras (esto podría realizarse con NLTK o spaCy también)
for _, row in df_rh[:None].iterrows():
    sentence_tokens.append(text_to_word_sequence(row[0]))
# Demos un vistazo
sentence_tokens[:2]

[['come', 'on', 'come', 'on'], ['you', 'think', 'you', 'drive', 'me', 'crazy']]

Cargamos el nuevo vocabulario, usamos los mismos parámetros del modelo y el mismo callback en training.

In [65]:

w2v_model.build_vocab(sentence_tokens)

In [66]:
w2v_model.corpus_count

2343

Verificamos que sobreescribió el anterior dado que ahora tiene el tamaño del corpus de Radiohead.

In [68]:

w2v_model.train(sentence_tokens, 
                   total_examples=w2v_model.corpus_count, 
                   epochs=35, # Aumentamos épocas para compensar el tamaño del corpus
                   compute_loss=True, 
                   callbacks=[callback()])

Loss after epoch 0: 21858.552734375
Loss after epoch 1: 20822.201171875
Loss after epoch 2: 20940.01171875
Loss after epoch 3: 20730.40625
Loss after epoch 4: 21327.9375
Loss after epoch 5: 20638.4140625
Loss after epoch 6: 20198.0859375
Loss after epoch 7: 19462.875
Loss after epoch 8: 19982.28125
Loss after epoch 9: 19437.359375
Loss after epoch 10: 19881.4375
Loss after epoch 11: 19816.5
Loss after epoch 12: 19151.375
Loss after epoch 13: 18699.9375
Loss after epoch 14: 19359.375
Loss after epoch 15: 18969.46875
Loss after epoch 16: 18978.3125
Loss after epoch 17: 18775.375
Loss after epoch 18: 18623.53125
Loss after epoch 19: 18627.125
Loss after epoch 20: 18575.875
Loss after epoch 21: 18884.40625
Loss after epoch 22: 18224.71875
Loss after epoch 23: 18144.40625
Loss after epoch 24: 18863.46875
Loss after epoch 25: 18529.09375
Loss after epoch 26: 18387.65625
Loss after epoch 27: 17855.375
Loss after epoch 28: 17777.3125
Loss after epoch 29: 17668.1875
Loss after epoch 30: 17206.7

(204889, 413105)

In [69]:
# Cantidad de words encontradas en el corpus
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))


Cantidad de words distintas en el corpus: 383


Observación del entrenamiento.

Radiohead converge más rápido y estable que los Beatles a pesar de tener menor vocabulario (383 vs 445 palabras), lo cual me sorprende en relación a lo abstracto de las letras de Radiohead.

Veamos algunas palabras similares a otras.

In [93]:
# Palabras MAS relacionadas con "dance"
w2v_model.wv.most_similar(positive=["dance"], topn=10)

[('uptight', 0.6517869234085083),
 ('lotus', 0.6507130265235901),
 ('remember', 0.6246110796928406),
 ('talk', 0.6174523830413818),
 ('burn', 0.6123154759407043),
 ('everyone', 0.6028121113777161),
 ('turn', 0.5873841643333435),
 ('denial', 0.5762035846710205),
 ('goes', 0.5606090426445007),
 ('eye', 0.5510299205780029)]

Parecen palabras en el contexto de la letra, más que palabras similares, de hecho las probabilidades de relación positiva no son muy altas.

In [72]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["love"], topn=10)


[('needed', 0.587641716003418),
 ('choke', 0.5561240911483765),
 ('uptight', 0.5487158298492432),
 ('can', 0.5194022059440613),
 ("payin'", 0.5176680088043213),
 ('turning', 0.5055940747261047),
 ('help', 0.49425676465034485),
 ('why', 0.4877564013004303),
 ('its', 0.48602384328842163),
 ('rain', 0.48233190178871155)]

De nuevo, de hecho love en los Beattles son más relacionadas por probabilidad.

In [85]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["come"], topn=5)

[('door', 0.6256129145622253),
 ('well', 0.6028443574905396),
 ('slowly', 0.5700554847717285),
 ('someone', 0.5436341762542725),
 ('begin', 0.5435992479324341)]

In [87]:
# Palabras que MENOS se relacionan con...:
w2v_model.wv.most_similar(negative=["creep"], topn=10)

[('needed', -0.11634914577007294),
 ('falling', -0.13417813181877136),
 ('mouth', -0.15238161385059357),
 ('try', -0.15574079751968384),
 ('what', -0.16551315784454346),
 ('fo', -0.16598688066005707),
 ('blow', -0.1727825105190277),
 ('into', -0.17543841898441315),
 ('my', -0.17930914461612701),
 ('why', -0.18484504520893097)]

In [90]:
# el método get_vector permite obtener los vectores:
vector_sleep = w2v_model.wv.get_vector("sleep")
#print(vector_sleep)
# el método most_similar también permite comparar a partir de vectores
w2v_model.wv.most_similar(vector_sleep)

[('sleep', 1.0),
 ('we', 0.6399301886558533),
 ('uptight', 0.6142443418502808),
 ('wings', 0.6109893918037415),
 ('tonight', 0.6086946725845337),
 ('stop', 0.5920105576515198),
 ('town', 0.5870482921600342),
 ('green', 0.5863350033760071),
 ('great', 0.5860240459442139),
 ('cards', 0.5784515142440796)]

In [91]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["sleep"], topn=10)


[('we', 0.6399301886558533),
 ('uptight', 0.6142443418502808),
 ('wings', 0.6109893918037415),
 ('tonight', 0.6086947321891785),
 ('stop', 0.592010498046875),
 ('town', 0.5870482921600342),
 ('green', 0.5863348841667175),
 ('great', 0.5860240459442139),
 ('cards', 0.5784515142440796),
 ('happen', 0.5658345222473145)]

los dos últimos son equivalentes.

La observación general es que las relaciones positivas se dan por el contexto y la temática de las letras.

In [98]:
# Reducción de dimensiones
vecs_rh, labels_rh = reduce_dimensions(w2v_model, num_dimensions=2)

# Visualización adecuada (MAX_WORDS ajustable para evitar solapamiento)
MAX_WORDS = 150
fig = px.scatter(x=vecs_rh[:MAX_WORDS,0], 
                 y=vecs_rh[:MAX_WORDS,1], 
                 text=labels_rh[:MAX_WORDS],
                 title="Visualización de Embeddings - Radiohead (2D)")
fig.update_traces(textposition='top center')
fig.show(renderer="colab")

Al observar el gráfico. 
La primera observación más clara es el grupo de preposiciones y artículos que aparecen en el cuadro superior izquierdo:
"a, the" y "in, into, out, down, on", "coming, around" parecieran indicar acciones simples posicionales, pero creo que en realidad son casi todas step words.
 
En el cuadro inferior "wish, want, love" parecieran indicar emociones o deseos y la luna entra ahí como símbolo.


![En el cuadro inferior "wish, want, love" parecieran indicar emociones o deseos](img/wwl.png)




A la derecha "lost, lies, mess, don't, hurt, but" tienen connotación negativa. Banda depresiva? XD.

Interesante esto en dos dimensiones.


![también se cuela alguna como arms](img/depressed.png)




En general se identifican algunas parejas de palabras que están en canciones, como "lost myself" en Karma Police.
